# Data Science Workflow

## SQL

You've seen how SQL, Structured Query Language, is a wonderful method for organizing data. Having data in a database allows multiple users to access, query, and update shared data simultaneously. As such, many companies store their data in databases and SQL is undoubtedly the most popular implementation. You saw that most statements began with the `SELECT` clause, and from there you specify which columns you wish to select, the table from which you wish to select them, and appropriate filters using the `WHERE` clause. You can also aggregate data with the `GROUP BY` clause, and apply further filters to those roll-ups using the `HAVING` clause. Furthermore, you can select data from multiple tables using a `JOIN`, so long as the two tables have a common key(s) which you specify with the `USING` clause, if the column names are identical, or more verbosely, you can specify how to join the tables with the `ON` clause. 

For example, here's the schema for the mock customer relationship database that you've seen before: 


<img src='assets/01_data_workflow/Database-Schema.png' width=550>

If you wanted to select some aggregate statistics regarding employees and customers on a per office basis for cities with at least 2 offices, you could write a query like this:  

```SQL
SELECT officeCode,
       city,
       COUNT(employeeNumber) AS num_employees,
       COUNT(customerNumber) AS num_customers,
       sum(amount) AS total_payments_received
       FROM offices
       JOIN employees
       USING(officeCode)
       JOIN customers
       ON employeeNumber = salesRepEmployeeNumber
       JOIN payments
       USING(customerNumber)
       GROUP BY 1, 2
       WHERE city IN (SELECT city FROM offices GROUP BY 1 HAVING COUNT(*)>1);
```

Look at that; you even got to see the use of a subquery once again!
       

## Data Exploration and Visualization

Once you've loaded in some data from a SQL database or elsewhere, it's generally time to start investigating and cleaning up that data. Remember that Pandas comes with many useful methods for quickly exploring your dataset. For example, you may want to initially check the datatypes of your dataset and how many null values exist for each of the various features. This is incredibly easy using the `.info()` method. Similarly, if you want to generate aggregate statistics, you can simply call the `.describe()` method on the DataFrame. If you want to quickly explore the distribution and pairwise correlations between your features, then using the `pd.plotting.scatter_matrix()` function will quickly display a grid of histograms and scatter plots for the various features; the diagonal entries are the distributions for each of the variables, while any other cell [i,j] is a scatter plot of feature i vs feature j.  

Recall that when initially exploring your data, you are both looking for insights and anomalies. Typically, you want to get familiar with the data, what it represents, and how it is represented. This will often give you further ideas about how to mine the data for structure and answer questions regarding the application.

## Regression

After acquiring and exploring your data (including cleaning it up), you'll then go on to model said data using the regression techniques you learned about earlier. With this, recall that there are four main assumptions underlying a linear regression model.

### 1. Linearity

With linear models, the target variable is being modeled as a linear combination of the independent variables. As such, there should be a linear relationship between the target variable and the various features being used. If the rate of change between the target variable and one of the features is non-linear and displays other characteristics such as an exponential acceleration, then prior transformations of the data are necessary before applying a regression model. 


### 2. Normality

With linear models, the errors (residuals) from the model are assumed to be normally distributed. A good heuristic to initially check for this is to use a Q-Q plot. 

### 3. Homoscedasticity

Along with the assumption of normal distribution, error terms should also not be correlated with the target variable or other features within the model. If errors indeed appear to be random and there are no discernible trends, then the errors are said to be homoscedastic. Looking at a simple plot of residuals against the target variable or other feature is generally sufficient to gauge this.

### 4. Independence

Finally, regression models assume that the various independent features feeding into the model are independent. You'll take a further look at this in this section and investigate how to check for multicollinearity. Multicollinearity is when a variable can be predicted with substantial accuracy by a separate set of features. Previously, you've examined multicollinearity in the context of the "dummy variable trap" and the two variable case. It's unwise to include two features in a regression model that are highly correlated. Similarly, in a multivariate case, having a set of features that can effectively predict another independent feature can be problematic. Such phenomenon will not reduce the overall accuracy of the model, but will **severely impede interpretation as coefficient weights of the model** become unstable so it is difficult or impossible to determine which features are most influential.

---

## Obtaining Your Data

In [ ]:
import sqlite3
import pandas as pd

# Create a connection
con = sqlite3.connect('data/obtaining_your_data/data.sqlite')
# Create a cursor
cur = con.cursor()
# Select some data
cur.execute("""SELECT * FROM orders JOIN orderdetails USING(orderNumber);""")
df = pd.DataFrame(cur.fetchall())
df.columns = [i[0] for i in cur.description]
print(df.shape)
df.head()

(2996, 11)


,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber,productCode,quantityOrdered,priceEach,orderLineNumber
0,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,,363,S18_1749,30,136.00,3
1,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,,363,S18_2248,50,55.09,2
2,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,,363,S18_4409,22,75.46,4
3,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,,363,S24_3969,49,35.29,1
4,10101,2003-01-09,2003-01-18,2003-01-11,Shipped,Check on availability.,128,S18_2325,25,108.06,4


In [ ]:
# Create a connection
con = sqlite3.connect('data/obtaining_your_data/data.sqlite')
# Create a cursor
cur = con.cursor()
# Select some data
cur.execute("""SELECT * FROM products;""")
df = pd.DataFrame(cur.fetchall())
df.columns = [i[0] for i in cur.description]
print(df.shape)
df.head()

(110, 9)


,productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; deta...,7305,98.58,214.30
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,1:10,Highway 66 Mini Classics,"Official Moto Guzzi logos and insignias, saddl...",6625,68.99,118.94
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,1:10,Red Start Diecast,"Model features, official Harley Davidson logos...",5582,91.02,193.66
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,1:10,Motor City Art Classics,Features include: Turnable front wheels; steer...,3252,85.68,136.00


## Merging Data

Recall, that you can also join data from multiple tables in SQL.

In [ ]:
# Create a connection
con = sqlite3.connect('data/obtaining_your_data/data.sqlite')
# Create a cursor
cur = con.cursor()
# Select some data
cur.execute("""SELECT * FROM products
                        JOIN orderdetails
                        USING (productCode);""")
df = pd.DataFrame(cur.fetchall())
df.columns = [i[0] for i in cur.description]
print(df.shape)
df.head()

(2996, 13)


,productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP,orderNumber,quantityOrdered,priceEach,orderLineNumber
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70,10107,30,81.35,2
1,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70,10121,34,86.13,5
2,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70,10134,41,90.92,2
3,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70,10145,45,76.56,6
4,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70,10159,49,81.35,14


You can also merge data from a separate csv file. For example, say you take a separate data source regarding daily sales data for the various branches. You might first generate a view from our database:

In [ ]:
# Create a connection
con = sqlite3.connect('data/obtaining_your_data/data.sqlite')
# Create a cursor
cur = con.cursor()
# Select some data
cur.execute("""SELECT * FROM customers
                        JOIN orders
                        USING(customerNumber);""")
df = pd.DataFrame(cur.fetchall())
df.columns = [i[0] for i in cur.description]
print(df.shape)
df.head()

(326, 19)


,customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit,orderNumber,orderDate,requiredDate,shippedDate,status,comments
0,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,France,1370,21000.00,10123,2003-05-20,2003-05-29,2003-05-22,Shipped,
1,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,France,1370,21000.00,10298,2004-09-27,2004-10-05,2004-10-01,Shipped,
2,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,France,1370,21000.00,10345,2004-11-25,2004-12-01,2004-11-26,Shipped,
3,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,,Las Vegas,NV,83030,USA,1166,71800.00,10124,2003-05-21,2003-05-29,2003-05-25,Shipped,Customer very concerned about the exact color ...
4,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,,Las Vegas,NV,83030,USA,1166,71800.00,10278,2004-08-06,2004-08-16,2004-08-09,Shipped,


And then load the seperate datefile:

In [ ]:
daily_sums = pd.read_csv('data/obtaining_your_data/Daily_Sales_Summaries.csv')
daily_sums.head()

,orderDate,min,max,sum,mean,std
0,2003-01-06,1660.12,4080.00,10223.83,2555.957500,1132.572429
1,2003-01-09,1463.85,4343.56,10549.01,2637.252500,1244.866467
2,2003-01-10,1768.33,3726.45,5494.78,2747.390000,1384.599930
3,2003-01-29,1283.48,5571.80,50218.95,3138.684375,1168.280303
4,2003-01-31,1338.04,4566.99,40206.20,3092.784615,1148.570425


In [ ]:
merged = pd.merge(df, daily_sums)

## Checking Merged Data

It's always good practice to check assumptions and preview transformed data views throughout your process. Let's take a look:

In [ ]:
merged.head()

,customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,...,orderDate,requiredDate,shippedDate,status,comments,min,max,sum,mean,std
0,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,...,2003-05-20,2003-05-29,2003-05-22,Shipped,,2163.50,5282.64,14571.44,3642.860000,1322.891537
1,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,...,2004-09-27,2004-10-05,2004-10-01,Shipped,,1938.24,4128.54,6066.78,3033.390000,1548.775983
2,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",,Nantes,,44000,...,2004-11-25,2004-12-01,2004-11-26,Shipped,,557.60,7573.50,20564.45,2570.556250,2178.832190
3,350,Marseille Mini Autos,Lebihan,Laurence,91.24.4555,"12, rue des Bouchers",,Marseille,,13008,...,2004-11-25,2004-12-02,2004-11-29,Shipped,,557.60,7573.50,20564.45,2570.556250,2178.832190
4,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,,Las Vegas,NV,83030,...,2003-05-21,2003-05-29,2003-05-25,Shipped,Customer very concerned about the exact color ...,798.38,4704.92,40207.06,2680.470667,1255.052262


Pandas' `merge()` function conveniently uses common column names between the DataFrames as keys. You can always specify what columns to join on by using the `on` keyword as in `pd.merge(df1, df2, on=[col1, col2])`. Unfortunately, columns that are not identically named beforehand will not work with this method. Additionally, it is imperative to check the formatting of the join keys between the tables. A number formatted as a string can often ruin joins, and separate formatting conventions such as 'U.S.' versus 'USA' are also important preprocessing considerations before merging data files from various sources. In this case, everything worked smoothly, but it's good to keep in mind what problems may occur.

## Saving Transformed Data to File
Finally, we can save our transformed dataset.

In [ ]:
merged.to_csv('data/obtaining_your_data/Merged_Dataset.csv', index=False)

---

## Practice: Obtaining Data

## Introduction
In this lab you'll practice your munging and transforming skills in order to load in your data to solve a regression problem.

## Objectives
You will be able to:
* Perform an ETL process with multiple tables and create a single dataset

## Task Description

You just got hired by Lego! Your first project is going to be to develop a pricing algorithm to help set a target price for new lego sets that are released to market. To do this, you're first going to need to start mining the company database in order to collect the information you need to develop a model.

Start by investigating the database stored in data/obtaining_your_data_lab/lego.db and joining the tables into a unified dataset!

> **Hint:** use this SQL query to preview the tables in an unknown database:
```sql
SELECT name FROM sqlite_master
            WHERE type='table'
            ORDER BY name;
```

In [ ]:
# Your code here

---

## Exploring Your Data

## Exploratory Data Analysis

Exploratory Data Analysis, or **_EDA_**, is a crucial part of any Data Science project.  Before you can go off building models on a dataset, you need to be familiar with the actual data it contains -- otherwise, you'll have no intuition about how to interpret the results of these models, or even if you can trust them at all!

This lesson will outline the basic steps that should be taken -- and questions that should be answered during EDA. 

## Understanding the Distribution of the Dataset

One of the foundational pieces of an EDA investigation is to understand the underlying distribution of the data.  Often, some of the most interesting/important business insights come not from machine learning models, but simply from exploring the distribution of the dataset! If your company or organization has not yet mastered reporting on descriptive analytics, the insights gained here can be invaluable to company strategy -- think questions such as "who is my most profitable customer segment?" or "is there a seasonality to our customer churn rate?".  These are important questions to any business, and they don't require machine learning models to answer them -- just some basic visualizations, and the ability to ask good questions.

Getting a feel for the distribution of a dataset is done in a few different ways. Generally, you'll make use of high-level descriptive statistics, followed by visualizations. During the EDA process, it is quite common to uncover interesting things in the data that spur further questions for the investigation.  

> "The most exciting phrase to hear in science, the one that heralds new discoveries, is not 'Eureka!' (I found it!) but 'That's funny...'"
>
>                                            - Isaac Asimov


Recall that Pandas can easily provide descriptive statistics of a DataFrame by using the DataFrame class's built-in `.describe()` method.  The resulting output is a table containing information such as the count, mean, median, min, max, and quartile values for every column in the DataFrame.  This is especially handy for answering questions such as "how much variance can I expect in column {X}?"

### Visualizing Distributions - Histograms

The easiest way to understand the distribution of a dataset is to visualize it! Recall that since `pandas` uses the `matplotlib` library, you can easily create histograms showing the distribution of each column by using the DataFrame's built-in `.hist()` method.  

<img src='assets/01_data_workflow/sample_hist.png'>

### Visualizing Distributions - Kernel Density Estimation (KDE) Plots

Another great way of quickly visualizing the distribution of a column is to construct a **_KDE Plot_**. This is often overlaid on a histogram to create a line that visualizes an approximate probability density of the variable. 

<img src='assets/01_data_workflow/sample_kde.png'>


### Using Joint Plots

A more advanced visualization tool you can make use of is the **_Joint Plot_**.  This allows you to visualize a scatterplot, the distributions of two different columns, a [KDE plot](https://seaborn.pydata.org/generated/seaborn.kdeplot.html), and even a simple regression line all on the same visualization. In practice, this is incredibly handy for doing this like checking the linearity assumption between predictors and a target variable during a regression analysis. 

Since joint plots are more advanced than a basic visualization like a histogram or scatterplot, you'll need to make use of the **_seaborn_** library to create them. The syntax for creating a joint plot is:

```python
# sns is the standard alias for seaborn
sns.jointplot(x= <column>, y= <column>, data=<dataset>, kind='reg')

```

<img src='assets/01_data_workflow/sample_jointplot.png'>

For full details on how to create joint plots with seaborn, see the [seaborn documentation on joint plots](https://seaborn.pydata.org/generated/seaborn.jointplot.html)!

## Interpreting Your EDA Results

It is worth noting that the goal of EDA is not pretty visualizations -- it's _insight into your data_!  Don't fall into the trap of thinking that EDA means building a couple of quick visualizations and then moving onto modeling -- you should actively try to generate questions and see if you can answer them by exploring the dataset.  Visualizations are great, but only because they make it easy to quickly interpret our data.  Use them as a tool, not a goal, during the EDA process!

---

## Practice: Exploratory Data Analysis

## Data Exploration

At this point, you've already done a modest amount of EDA between investigating the initial dataset to further exploring individual features while cleaning things up in preparation for modeling. During this process, you've become more familiar with the particular idiosyncrasies of the dataset. This gives you an opportunity to uncover difficulties and potential pitfalls in working with the dataset as well as potential avenues for feature engineering that could improve the predictive performance of your model down the line. Remember that this is also not a linear process; after building an initial model, you might go back and continue to mine the dataset for potential inroads to create additional features and improve the model's performance if the initial results did not satisfy your needs and expectations. Here, you'll continue this process, investigating the distributions of some of the various features and their relationship to the target variable: `list_price`.

In the cells below: 

* Import `pandas` and set the standard alias. 
* Import `numpy` and set the standard alias. 
* Import `matplotlib.pyplot` and set the standard alias. 
* Import `seaborn` and set the alias `sns` (this is the standard alias for seaborn). 
* Use the ipython magic command to set all matplotlib visualizations to display inline in the notebook. 
* Load the dataset stored in the `'Lego_data_cleaned.csv'` file into a DataFrame, `df`. 
* Inspect the head of the DataFrame to ensure everything loaded correctly.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import libraries and load Lego_data_merged.csv

- Describe the dataset using 5-point statistics.

In [ ]:
# Your code here

- Use pandas to plot histograms for all the numeric variables in the dataset.

In [ ]:
# Your code here

Note how skewed most of these distributions are. While linear regression does not assume that each of the individual predictors are normally distributed, it does assume a linear relationship between the predictors and the target variable (`list_price` in this case). To further investigate if this assumption holds true, you can plot some single variable regression plots of each feature against the target variable using `seaborn`. 

## Check for Linearity

Recall that one assumption in linear regression is that the target variable is linearly related to the input features. As shown in the previous lesson, you can use the `sns.jointplot()` function to investigate whether this relation holds true for the various predictors on hand.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here

## Checking for Multicollinearity

It's also important to make note of whether your predictive features will result in multicollinearity in the resulting model. While definitive checks for multicollinearity require analyzing the resulting model, predictors with overly high pairwise-correlation (r > .65) are almost certain to produce multicollinearity in a model. With that, take a minute to generate the pairwise (pearson) correlation coefficients of your predictive features and visualize these coefficients as a heatmap.

In [ ]:
# Your code here

In [ ]:
# Your code here

---

## Scrubbing and Cleaning Data

## The "Scrub" Step

During the process of working with data, you'll always reach a point where you've gathered all the data you'll need (our "Obtain" step), but the data is not yet in a format where you can use it for modeling.  All the work that you'll be doing in the next lab will be to get the dataset in a format that you can easily explore and build models with. 


## Subsampling to Reduce Size

When building a model for predictive purposes, more data is always better when training the final model. However, during the development process when working with large datasets, it is common to work with only a subsample of the dataset. Building a model is an iterative process -- often, you fit the model, investigate the results, then train the model again with some small tweaks based on what you noticed.  Since this is an iterative process, you want to avoid long runtimes, and iterate as quickly as possible.  When you're satisfied with the model you've built on the subsample of data, then you would fit the model on the entire dataset. 

In the next lab, you'll work with a subsample of the dataset to increase the iteration speed in refining your model. 


## Dealing With Data Types


One of the most common problems you'll need to deal with during the data scrubbing step is columns that are encoded as the wrong data type. For example, a common formatting issue you may encounter is numeric data that is mistakenly encoded as string data. This makes numerical operations on the data impossible without first reformatting the data. A simple operation such as `2 + 2` will return `22` if the numeric data is accidentally formatted as strings (`'2'+'2'='22'`). Similarly, categorical data is often encoded as integer values. If you don't properly conceptualize how the data is represented, you will fail to formulate meaningful models and insights.

A first step to uncover and investigate such issues is to use the `.info()` method available for all Pandas DataFrames. This will tell what type of data each column contains, as well as the number of values contained within that column (which can also help us identify columns that contain missing data)!  Here's an example response:

```
<class 'pandas.core.frame.DataFrame'>
Int64Index: 97839 entries, 0 to 97838
Data columns (total 16 columns):
Store           97839 non-null object
Dept            97839 non-null object
Date            97839 non-null object
Weekly_Sales    97839 non-null float64
IsHoliday       97839 non-null bool
Type            97839 non-null object
Size            97839 non-null int64
Temperature     97839 non-null float64
Fuel_Price      97839 non-null float64
MarkDown1       35013 non-null float64
MarkDown2       27232 non-null float64
MarkDown3       32513 non-null float64
MarkDown4       34485 non-null float64
MarkDown5       35013 non-null float64
CPI             97839 non-null float64
Unemployment    97839 non-null float64
dtypes: bool(1), float64(10), int64(1), object(4)
memory usage: 12.0+ MB

```

From here, a good next step would be to look at examples from each column encoded as strings (remember, Pandas refers to string columns as `object`) and confirm that this data is supposed to be encoded as strings. One method to do this is to preview a truncated version of the output from `.value_counts()`. For example, you could preview the 5 most frequent entries from each column with a simple loop like this:  

```python
for col in df.columns:
    try:
        print(col, df[col].value_counts()[:5])
    except:
        print(col, df[col].value_counts())
        # If there aren't 5+ unique values for a column the first print statement
        # will throw an error for an invalid idx slice
    print('\n') # Break up the output between columns
```

It is usually also a good idea to check integer columns to ensure that the data it contains is meant to represent actual numeric data, and is not just categorical data encoded as integers. You may also uncover null values hard coded as strings such as `"?"`, `"999999"` or other extraneous values depending on the dataset and who created it.

### Numeric Data Encoded as Strings

If you've identified numeric data encoded as strings, it's typically a pretty easy problem to solve. Often it's as simple as casting the string data to a numeric type:

```python
df['numeric_string_col'] = df['numeric_string_col'].astype('float')
```

Sadly, it's not always that simple. For example, if there is even a single cell that contains a letter or non-numeric character such as a comma or monetary symbol ($) the above statement will fail. In such cases, a more complex cleaning function must be manually created. This could involve stripping extraneous symbols such as ',/$/%',  or simply casting non convertible strings as null. Recall that when NumPy sees multiple data types in an array, it defaults to casting everything as a string. If you try to cast a column from string to numeric data types and get an error, consider checking the unique values in that column -- it's likely that you may have a single letter hiding out somewhere that needs to be removed!

### Categorical Data Encoded as Integers

It's also common to see categorical data encoded as integers.  Given that a big step in the data cleaning process is to convert all categorical columns to numeric equivalents, this may not seem like a problem at first glance.  However, leaving categorical data encoded as integers can have a negative effect by introducing bad information into our model. This is because integer encoding mistakenly adds mathematical relationships between the different categories -- our model may mistakenly think that the category represented by the integer `4` twice as much as category `2`, and so on.  

The best way of dealing with this problem is to cast the entire column to a string data type, which will better represent the column's categorical nature. Since it's categorical, we can correctly deal with it when we one-hot encode categorical data later in the process.

The following example shows the syntax necessary for converting a column from one data type to another:

```python
# Cast to a numeric type
df['some_column'] = df.['some_column'].astype('float32')

# Cast back to a string type
df['some_column'] = df.['some_column'].astype('str')
```

Once done, it is then common to pass these categorical variables to another function such as `pd.get_dummies()` in order to transform these features into representations that are more suitable for machine learning algorithms. It may be necessary to drop the first dummy to avoid the dummy variable trap.

## Detecting and Dealing With Null Values

Another important data cleaning check is to inspect for missing or null values. Recall from previous labs that Pandas denotes missing values as `NaN`.

### Checking For `NaN`s

You can easily check how many missing values are contained within each column by having Pandas create a truth table where the cells that contain `NaN` are marked as `True` and everything else is marked as `False`.

```python
# Create a truth table for missing values
df.isna()
```

Since `False=0` and `True=1` in programming, you can then `sum()` these truth tables to get a column-by-column count of the number of missing values in the dataset. 

```python 
# Check how many missing values in each column
df.isna().sum()
```

As noted above, remember that your dataset may also contain null values that are denoted by placeholder values.  Most datasets that do this will make mention of this in the dataset's data dictionary. However, you may also see these denoted by extreme values that don't make sense (e.g. a person's weight being set to something like 0 or 10000).  Doing a quick manual inspection of the top values for each feature is often the only manner to detect such anomalies.

### Dealing With Null Values

There are several options for dealing with null values. You can always remove observation rows with missing values or similarly remove features with excessive sparsity caused by null values. That said, doing so throws away potentially valuable information. There may be important reasons why said information is missing. Despite this, many machine learning algorithms will not tolerate null values and as such you either have to impute values or drop the data. Some options you have for imputing data include:

**_Numeric Data_**
* Replacing Nulls with the column median
* Binning the data and converting columns to a categorical format (_Coarse Classification)_

**_Categorical Data_**
* Making Null values their own category
* Replacing null values with the most common category 

As a data scientist, you will have to decide which method of imputation is appropriate for your particular data set and business objective.

## Checking For Multicollinearity

Before proceeding to modeling, you also want to check that the data does not have high multicollinearity or correlation/covariance between predictor columns.  

The easiest way to do this to build and interpret a correlation heatmap with the `seaborn` package. 

<img src='assets/01_data_workflow/heatmap.png'>

Columns with strong correlation should be dealt with by removing one of the offending columns, or by combining the columns through feature engineering (more on this later in the curriculum). After all, highly correlated features makes feature weights unstable and also impede model interpretability. That said, they are not apt to reduce model performance if that is the sole consideration.

The [seaborn documentation](https://seaborn.pydata.org/examples/many_pairwise_correlations.html) provides a great example code on how to build a correlation heatmap with data stored in a Pandas DataFrame. 


## Normalizing Data

An important step during the data cleaning process is to convert all of our data to the same scale by **_normalizing_** it.  

The most common form of data normalization is by converting data to z-scores. This is commonly referred to as standardization.

$$ \Large z= \dfrac{x-\mu}{\sigma}$$

$$ \large \mu = \text{Mean}$$
$$ \large \sigma = \text{Standard Deviation}$$

There are also other sorts of scaling methods we can use, such as **_Min-Max normalization_**:

$$\large z= \dfrac{x-\min(x)}{\max(x)-\min(x)}$$

In practice, z-score normalization is the most widely used.

---

## Practice: Scrubbing Data

## Getting Started

You'll find the resulting dataset from your work in the _Obtaining Data_ Lab stored within the file `'data/scrubbing_and_cleaning_data_lab/Lego_data_merged.csv'`.  

In the cells below:

* Import `pandas` and set the standard alias. 
* Import `numpy` and set the standard alias. 
* Import `matplotlib.pyplot` and set the standard alias. 
* Import `seaborn` and set the alias `sns` (this is the standard alias for seaborn). 
* Use the ipython magic command to set all matplotlib visualizations to display inline in the notebook. 
* Load the dataset stored in the `'data/scrubbing_and_cleaning_data_lab/Lego_data_merged.csv'` file into a DataFrame, `df`. 
* Inspect the head of the DataFrame to ensure everything loaded correctly.

In [ ]:
# Import statements go here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Now, load in the dataset and inspect the head to make sure everything loaded correctly
df = pd.read_csv('data/scrubbing_and_cleaning_data_lab/Lego_data_merged.csv')

In [ ]:
display(df.columns)
display(df.sample(3), len(df))

Index(['prod_id', 'ages', 'piece_count', 'set_name', 'prod_desc',
       'prod_long_desc', 'theme_name', 'country', 'list_price', 'num_reviews',
       'play_star_rating', 'review_difficulty', 'star_rating',
       'val_star_rating'],
      dtype='object')

,prod_id,ages,piece_count,set_name,prod_desc,prod_long_desc,theme_name,country,list_price,num_reviews,play_star_rating,review_difficulty,star_rating,val_star_rating
5370,75529,8-14,92,Elite Praetorian Guard,Protect the First Order with the Elite Praetor...,Defend the upper echelon of the First Order wi...,Star Wars™,ES,$32.9278,7.0,4.1,Easy,4.7,3.9
3197,41193,8-12,451,Aira & the Song of the Wind Dragon,Join forces with Aira and Lumia to protect the...,"Seek the magic within to help Aira, Lumia and ...",Elves,CH,$50.898,2.0,4.5,Average,5.0,4.5
5958,60188,7-12,883,Mining Experts Site,Head to the LEGO® City mine and dig up some gold!,Grab your hard hat and head out to the LEGO® C...,City,FR,$115.88779999999998,1.0,5.0,NaN,5.0,5.0


10870

## Starting our Data Cleaning

To start, you'll deal with the most obvious issue: data features with the wrong data encoding.

### Checking Data Types

In the cell below, use the appropriate method to check the data type of each column.

In [ ]:
# Your code here
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10870 entries, 0 to 10869
Data columns (total 14 columns):
prod_id              10870 non-null int64
ages                 10870 non-null object
piece_count          10870 non-null int64
set_name             10870 non-null object
prod_desc            10512 non-null object
prod_long_desc       10870 non-null object
theme_name           10870 non-null object
country              10870 non-null object
list_price           10870 non-null object
num_reviews          9449 non-null float64
play_star_rating     9321 non-null float64
review_difficulty    9104 non-null object
star_rating          9449 non-null float64
val_star_rating      9301 non-null float64
dtypes: float64(4), int64(2), object(8)
memory usage: 1.2+ MB


Now, investigate some of the unique values inside of the `list_price` column.

In [ ]:
# Your code here
df.list_price.unique()

array(['$29.99', '$19.99', '$12.99', '$99.99', '$79.99', '$59.99',
       '$49.99', '$39.99', '$34.99', '$159.99', '$9.99', '$199.99',
       '$149.99', '$119.99', '$89.99', '$69.99', '$24.99', '$15.99',
       '$14.99', '$6.99', '$16.99', '$7.99', '$4.99', '$2.49', '$369.99',
       '$169.99', '$279.99', '$249.99', '$239.99', '$139.99', '$269.99',
       '$129.99', '$44.99', '$11.99', '$349.99', '$5.99', '$109.99',
       '$54.99', '$32.99', '$197.99', '$88.99', '$41.99', '$31.99',
       '$26.99', '$21.99', '$3.99', '$299.99', '$754.99', '$484.99',
       '$36.99', '$789.99', '$499.99', '$84.99', '$799.99', '$289.99',
       '$179.99', '$113.9924', '$75.9924', '$60.7924', '$53.1924',
       '$45.59240000000001', '$37.9924', '$189.9924', '$30.3924',
       '$22.7924', '$12.1524', '$227.9924', '$174.7924', '$121.5924',
       '$34.1924', '$18.9924', '$17.4724', '$13.6724', '$7.5924',
       '$15.1924', '$9.8724', '$6.0724', '$2.2724', '$91.1924',
       '$379.9924', '$303.9924000000001

### Numerical Data Stored as Strings

A common issue to check for at this stage is numeric columns that have accidentally been encoded as strings. For example, you should notice that the `list_price` column above is currently formatted as a string and contains a proceeding '$'. Remove this and convert the remaining number to a `float` so that you can later model this value. After all, your primary task is to generate model to predict the price.

> Note: While the data spans a multitude of countries, assume for now that all prices have been standardized to USD.

In [ ]:
# Your code here

### Detecting and Dealing With Null Values

Next, it's time to check for null values. How to deal with the null values will be determined by the columns containing them, and how many null values exist in each.  
 
In the cell below, get a count of how many null values exist in each column in the DataFrame.

In [ ]:
# Your code here

Now, get some descriptive statistics for each of the columns. You want to see where the minimum and maximum values lie.

In [ ]:
# Your code here

Now that you have a bit more of a understanding of each of these features you can make an informed decision about the best strategy for dealing with the various null values. 

Some common strategies for filling null values include:
* Using the mean of the feature
* Using the median of the feature
* Inserting a random value from a normal distribution with the mean and std of the feature
* Binning

Given that most of the features with null values concern user reviews of the lego set, it is reasonable to wonder whether there is strong correlation between these features in the first place. Before proceeding, take a minute to investigate this hypothesis.

In [ ]:
# Investigate whether multicollinearity exists between the review features 
# (num_reviews, play_star_rating, star_rating, val_star_rating)

Note that there is substantial correlation between the `play_star_rating`, `star_rating` and `val_star_rating`. While this could lead to multicollinearity in your eventual regression model, it is too early to clearly determine this at this point. Remember that multicollinearity is a relationship between 3 or more variables while correlation simply investigates the relationship between two variables.

Additionally, these relationships provide an alternative method for imputing missing values: since they appear to be correlated, you could use these features to help impute missing values in the others features. For example, if you are missing the `star_rating` for a particular row but have the `val_star_rating` for that same entry, it seems reasonable to assume that it is a good estimate for the missing `star_rating` value as they are highly correlated. That said, doing so does come with risks; indeed you would be further increasing the correlation between these features which could further provoke multicollinearity in the final model.

Investigate if you could use one of the other star rating features when one is missing. How many rows have one of `play_star_rating`, `star_rating` and `val_star_rating` missing, but not all three.

In [ ]:
# Your code here
# Number missing all three: 1421

Well, it seems like when one is missing, the other two are also apt to be missing. While this has been a bit of an extended investigation, simply go ahead and fill the missing values with that feature's median. Fill in the missing values of `review_difficulty` feature  with string `'unknown'`.

In [ ]:
# Your code here

## Normalizing the Data

Now, you'll need to convert all of our numeric columns to the same scale by **_normalizing_** our dataset.  Recall that you normalize a dataset by converting each numeric value to it's corresponding z-score for the column, which is obtained by subtracting the column's mean and then dividing by the column's standard deviation for every value. 


In the cell below:

* Normalize the numeric X features by subtracting the column mean and dividing by the column standard deviation. 
(Don't bother to normalize the `list_price` as this is the feature you will be predicting.)

In [ ]:
# Your code here

## Saving Your Results

While you'll once again practice one-hot encoding as you would to preprocess data before fitting a model, saving such a reperesentation of the data will eat up additional disk space. After all, a categorical variable with 10 bins will be transformed to 10 seperate features when passed through `pd.get_dummies()`. As such, while further practice is worthwhile, save your DataFrame as-is for now.

In [ ]:
# Your code here

## One-Hot Encoding Categorical Columns

As a final step, you'll need to deal with the categorical columns by **_one-hot encoding_** them into binary variables via the `pd.get_dummies()` function.  

When doing this, you may also need to subset the appropriate features to avoid encoding the wrong data. The `get_dummies()` function by default converts all columns with *object* or *category* dtype. However, you should always check the result of calling `get_dummies()` to ensure that only the categorical variables have been transformed. Consult the [documentation](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.get_dummies.html) for more details. If you are ever unsure of the data types, call the `.info()` method.

In the cell below, subset to the appropriate predictive features and then use `pd.get_dummies()` to one-hot encode the dataset properly.

In [ ]:
# Your code here